1. ĐIỀU KIỆN VÀO LỆNH (ENTRY LONG)
Chỉ mở lệnh Mua khi hội đủ 4 yếu tố cùng lúc:

Giá > Mây Kumo: Xác nhận xu hướng dài hạn đang Tăng.

Tenkan > Kijun: Xác nhận xu hướng ngắn hạn đang Tăng.

Chikou > Giá của 26 phiên trước: Đường trễ báo hiệu không có cản phía trên.

MACD cắt lên Signal: Xác nhận xung lực mua vừa được bơm vào.

2. ĐIỀU KIỆN THOÁT LỆNH (GỘP CHUNG SL & TP)
Sử dụng duy nhất ATR Trailing Stop (Khuyến nghị dùng hệ số 2.5 x ATR):

SL ban đầu: Đặt tại Giá Mua - (2.5 x ATR).

Dời SL (Trailing): Kéo mức bảo vệ lên theo công thức Đỉnh cao nhất từ lúc mua - (2.5 x ATR).

Quy tắc cốt lõi: Đường Trailing Stop này chỉ được dời lên, tuyệt đối không dời xuống. Cứ để mặc cho lợi nhuận chạy tự do.

Đóng lệnh (Exit): Chốt toàn bộ vị thế ngay lập tức khi giá quay đầu giảm và đâm thủng mức Trailing Stop hiện tại.

In [11]:
# Install required libraries
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# GLOBAL CONFIGURATION
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT'],
    'INTERVAL': '1h',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# MODULE 1: DATA HANDLER
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                print("US IP detected. Switching to Binance.US...")
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
            if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# MODULE 2: STRATEGY LOGIC (Ichimoku + MACD + ATR)
class IchimokuMACDStrategy:
    def __init__(self):
        self.atr_multiplier = 2.5
        self.atr_period = 14

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        # 1. Ichimoku Cloud
        df['Tenkan'] = (df['High'].rolling(9).max() + df['Low'].rolling(9).min()) / 2
        df['Kijun'] = (df['High'].rolling(26).max() + df['Low'].rolling(26).min()) / 2
        df['Senkou_A'] = ((df['Tenkan'] + df['Kijun']) / 2).shift(26)
        df['Senkou_B'] = ((df['High'].rolling(52).max() + df['Low'].rolling(52).min()) / 2).shift(26)
        df['Close_26_ago'] = df['Close'].shift(26) # Chikou Span logic

        # 2. MACD
        macd = ta.macd(df['Close'], fast=12, slow=26, signal=9)
        df['MACD'] = macd['MACD_12_26_9']
        df['Signal_Line'] = macd['MACDs_12_26_9']

        # 3. ATR
        df['ATR'] = ta.atr(df['High'], df['Low'], df['Close'], length=self.atr_period)

        df.dropna(inplace=True)
        return df

# MODULE 3: BACKTEST ENGINE (TRAILING STOP)
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0
        self.entry_price = 0.0
        self.trades =[]

        # Biến phục vụ Trailing Stop
        self.trailing_stop = 0.0
        self.highest_high = 0.0

        self.equity_curve = np.zeros(len(self.df))

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values
        times = self.df.index

        tenkan = self.df['Tenkan'].values
        kijun = self.df['Kijun'].values
        senkou_a = self.df['Senkou_A'].values
        senkou_b = self.df['Senkou_B'].values
        close_26_ago = self.df['Close_26_ago'].values

        macd = self.df['MACD'].values
        signal = self.df['Signal_Line'].values
        atr = self.df['ATR'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']

        s = self.strategy

        self.equity_curve[0] = self.capital
        self.equity_curve[1] = self.capital

        for i in range(2, len(self.df)):
            # EXIT LOGIC (ATR Trailing Stop)
            if self.position == 1:
                # Nếu giá đâm thủng đường Trailing Stop
                if L[i] <= self.trailing_stop:
                    # Chốt lệnh (Lấy giá thấp hơn giữa Open và Trailing Stop đề phòng Gap Down)
                    real_exit = min(O[i], self.trailing_stop) * (1 - slip)

                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': 'Trailing Stop Hit',
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    self.position = 0
                    self.invested_amount = 0.0
                else:
                    # Nếu an toàn, dời Trailing Stop lên theo Đỉnh cao nhất
                    self.highest_high = max(self.highest_high, H[i])
                    new_ts = self.highest_high - (s.atr_multiplier * atr[i-1])
                    self.trailing_stop = max(self.trailing_stop, new_ts) # Tuyệt đối chỉ dời LÊN

            # ENTRY LOGIC
            if self.position == 0:
                cloud_top = max(senkou_a[i-1], senkou_b[i-1])

                cond_1 = C[i-1] > cloud_top                     # Giá > Mây Kumo
                cond_2 = tenkan[i-1] > kijun[i-1]               # Tenkan > Kijun
                cond_3 = C[i-1] > close_26_ago[i-1]             # Chikou > Giá 26 phiên trước
                cond_4 = (macd[i-1] > signal[i-1]) and (macd[i-2] <= signal[i-2]) # MACD cắt lên Signal

                if cond_1 and cond_2 and cond_3 and cond_4:
                    self.position = 1
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

                    # Khởi tạo Trailing Stop
                    self.highest_high = H[i]
                    self.trailing_stop = self.entry_price - (s.atr_multiplier * atr[i-1])

                    # KIỂM TRA INTRA-BAR SL: Ngay cây nến mua nếu giá sập mạnh quét luôn SL
                    if L[i] <= self.trailing_stop:
                        real_exit = self.trailing_stop * (1 - slip)
                        pnl_pct = (real_exit - self.entry_price) / self.entry_price
                        pnl_amt = self.invested_amount * (pnl_pct - fee * 2)
                        self.capital += pnl_amt

                        self.trades.append({
                            'Date': times[i], 'Type': 'Long', 'Reason': 'Instant SL',
                            'Entry': self.entry_price, 'Exit': real_exit,
                            'PnL_Amt': pnl_amt, 'Equity': self.capital
                        })
                        self.position = 0
                        self.invested_amount = 0.0
                    else:
                        # Nâng TS nếu nến Entry đẩy giá lên cao
                        self.trailing_stop = max(self.trailing_stop, self.highest_high - (s.atr_multiplier * atr[i-1]))

            # CAL UNREALIZED PNL
            current_equity = self.capital
            if self.position == 1:
                current_value = self.invested_amount * (C[i] / self.entry_price)
                unrealized_pnl = current_value - self.invested_amount
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# MODULE 4: PERFORMANCE EVALUATOR
class Evaluator:
    @staticmethod
    def generate_report(equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        # TÍNH SHARPE RATIO THEO TIMEFRAME
        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Ichimoku + MACD + TS")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        # Plotly Implementation - Light/White Theme with Subplots for Drawdown
        fig = make_subplots(
            rows=2, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.08,
            row_heights=[0.7, 0.3]
        )

        # Trace 1: Equity Curve
        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=1, col=1)

        # Trace 2: Drawdown Area
        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=2, col=1)

        fig.update_layout(
            title=f"Portfolio Performance: {symbol} ({interval})",
            template="plotly_white",
            height=650,
            margin=dict(l=40, r=40, t=60, b=40),
            showlegend=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=1, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=2, col=1)

        fig.show()

# MAIN EXECUTION
if __name__ == "__main__":
    # Loop through multiple symbols
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = IchimokuMACDStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                equity,
                trades,
                CONFIG['INITIAL_CAPITAL'],
                symbol,
                CONFIG['INTERVAL'],
                CONFIG['POSITION_SIZE_PCT']
            )


---> Starting Backtest for: BTCUSDT <---
Checking connection for BTCUSDT...
Total: 8783 candles fetched.

========== PERFORMANCE REPORT: BTCUSDT (1h) ==========
           Strategy: Ichimoku + MACD + TS
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $10,471.23
            Net PnL: $471.23
       Total Return: 4.71%
      CAGR (Annual): 4.75%
       Max Drawdown: -0.72%
       Sharpe Ratio: 2.85
       Total Trades: 63
            Winrate: 47.62%
      Profit Factor: 2.53




---> Starting Backtest for: ETHUSDT <---
Checking connection for ETHUSDT...
Total: 8783 candles fetched.

========== PERFORMANCE REPORT: ETHUSDT (1h) ==========
           Strategy: Ichimoku + MACD + TS
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $9,909.30
            Net PnL: $-90.70
       Total Return: -0.91%
      CAGR (Annual): -0.91%
       Max Drawdown: -1.33%
       Sharpe Ratio: -0.70
       Total Trades: 60
            Winrate: 31.67%
      Profit Factor: 0.76

